# Semiconductor Image Restoration — NAFNet Baseline Pipeline

This notebook implements a complete, self-contained **NAFNet Baseline** pipeline for Joint 	imes$ Super-Resolution and Denoising of semiconductor inspection images.

### Dataset Properties Summary:
- **Input (NoisyLR)**:  	imes 128$ spatial dimensions, , noisy range $approx [-0.05, 1.41]$.
- **Target (Ground Truth GT)**:  	imes 256$ spatial dimensions, , clean normalized range 1.$.
- **Goal**: Map noisy low-resolution inputs $ to restored high-resolution images $.

## 1. System Setup & Environment Check

In [8]:
import os
import glob
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm.auto import tqdm
from pathlib import Path

# Hardware Acceleration Check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Operating on device: {device}")
if device.type == 'cuda':
    print(f"[INFO] GPU Device Name: {torch.cuda.get_device_name(0)}")

# Optional Google Drive Mount
# try:
#     from google.colab import drive
#     drive.mount('/content/drive')
#     print("[INFO] Google Drive mounted successfully.")
# except Exception:
#     print("[INFO] Running in local environment / non-Colab context.")


[INFO] Operating on device: cuda
[INFO] GPU Device Name: NVIDIA GeForce RTX 3060 Laptop GPU


## 2. Global Pipeline Configuration & Experiment Hyperparameters

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. GLOBAL PIPELINE CONFIGURATION & EXPERIMENT HYPERPARAMETERS
# ─────────────────────────────────────────────────────────────────────────────

# ── Experiment Seed & Determinism ─────────────────────────────────────────────
SEED = 42

# ── Robust Dataset Path Resolution (works locally from any CWD) ──────────────
# Walks up the directory tree to find the project root containing main_dataset/
def _find_project_root() -> Path:
    curr = Path.cwd().resolve()
    while curr != curr.parent:
        if (curr / "main_dataset").exists():
            return curr
        curr = curr.parent
    return Path.cwd().resolve()

PROJECT_ROOT = _find_project_root()

# ── Dataset Paths ─────────────────────────────────────────────────────────────
DATASET_ROOT = str(PROJECT_ROOT / "main_dataset" / "train" / "train")
LR_DIR       = os.path.join(DATASET_ROOT, "NoisyLR")
GT_DIR       = os.path.join(DATASET_ROOT, "GT")

# ── Model Naming & Versioning ─────────────────────────────────────────────────
MODEL_NAME    = "nafnet"
MODEL_VERSION = "nafnet1"
LATEST_SAVE   = "latest"

# ── Output Directories (relative to project root, tracked by git) ─────────────
OUTPUT_DIR = str(PROJECT_ROOT / "models")

# ── Latest Checkpoint Paths (overwritten on every run) ────────────────────────
MODEL_SAVE_DIR        = os.path.join(OUTPUT_DIR, LATEST_SAVE, MODEL_NAME)
CHECKPOINT_BEST_PATH  = os.path.join(MODEL_SAVE_DIR, f"{MODEL_NAME}_best.pth")
CHECKPOINT_FINAL_PATH = os.path.join(MODEL_SAVE_DIR, f"{MODEL_NAME}_final.pth")

# ── Versioned Checkpoint Paths (one file per version, never overwritten) ──────
VERSION_SAVE_DIR        = os.path.join(OUTPUT_DIR, MODEL_NAME)
VERSION_BEST_SAVE_PATH  = os.path.join(VERSION_SAVE_DIR, f"{MODEL_VERSION}_best.pth")
VERSION_FINAL_SAVE_PATH = os.path.join(VERSION_SAVE_DIR, f"{MODEL_VERSION}_final.pth")
VERSION_SAVE            = os.path.join(VERSION_SAVE_DIR, f"{MODEL_VERSION}.pth")

# ── Create all output directories ─────────────────────────────────────────────
for d in [MODEL_SAVE_DIR, VERSION_SAVE_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Hyperparameters & Training Settings ───────────────────────────────────────
BATCH_SIZE     = 8
EPOCHS         = 15
LEARNING_RATE  = 1e-3
WEIGHT_DECAY   = 1e-4
GRAD_CLIP_NORM = 1.0

# ── Set Deterministic Random Seeds ────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print("[INFO] Pipeline Configuration Initialized:")
print(f"  PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"  SEED          : {SEED}")
print(f"  DATASET_ROOT  : {DATASET_ROOT}")
print(f"  LR_DIR        : {LR_DIR}")
print(f"  GT_DIR        : {GT_DIR}")
print(f"  MODEL_SAVE_DIR: {MODEL_SAVE_DIR}  (latest)")
print(f"  VERSION_DIR   : {VERSION_SAVE_DIR}  (versioned)")
print(f"  MODEL         : {MODEL_NAME}  |  VERSION: {MODEL_VERSION}")
print(f"  BATCH_SIZE={BATCH_SIZE}, EPOCHS={EPOCHS}, LR={LEARNING_RATE}, WD={WEIGHT_DECAY}, GRAD_CLIP={GRAD_CLIP_NORM}")

[INFO] Pipeline Configuration Initialized:
  PROJECT_ROOT  : /home/mukundvinayak/semiconductor-image-restoration
  SEED          : 42
  DATASET_ROOT  : /home/mukundvinayak/semiconductor-image-restoration/main_dataset/train/train
  LR_DIR        : /home/mukundvinayak/semiconductor-image-restoration/main_dataset/train/train/NoisyLR
  GT_DIR        : /home/mukundvinayak/semiconductor-image-restoration/main_dataset/train/train/GT
  MODEL_SAVE_DIR: /home/mukundvinayak/semiconductor-image-restoration/models/latest/nafnet  (latest)
  VERSION_DIR   : /home/mukundvinayak/semiconductor-image-restoration/models/nafnet  (versioned)
  MODEL         : nafnet  |  VERSION: nafnet1
  BATCH_SIZE=8, EPOCHS=15, LR=0.001, WD=0.0001, GRAD_CLIP=1.0


## 3. Dataset Path Verification

In [10]:
if not os.path.exists(LR_DIR) or not os.path.exists(GT_DIR):
    raise FileNotFoundError(f"[ERROR] Dataset directory not found at '{LR_DIR}' or '{GT_DIR}'.")

lr_count = len(glob.glob(os.path.join(LR_DIR, "*.npy")))
gt_count = len(glob.glob(os.path.join(GT_DIR, "*.npy")))
print(f"[INFO] Dataset paths verified: {lr_count} NoisyLR files, {gt_count} GT files in '{DATASET_ROOT}'.")

[INFO] Dataset paths verified: 3200 NoisyLR files, 3200 GT files in '/home/mukundvinayak/semiconductor-image-restoration/main_dataset/train/train'.


## 4. PyTorch Dataset & Data Loader

In [11]:
class PairedNpyDataset(Dataset):
    def __init__(self, lr_dir: str, gt_dir: str, augment: bool = True):
        super().__init__()
        self.lr_filenames = sorted(glob.glob(os.path.join(lr_dir, "*.npy")))
        self.gt_filenames = sorted(glob.glob(os.path.join(gt_dir, "*.npy")))
        self.augment = augment
        assert len(self.lr_filenames) == len(self.gt_filenames), "Mismatch between LR and GT files count"
        for lr, gt in zip(self.lr_filenames, self.gt_filenames):
            assert Path(lr).stem == Path(gt).stem, f"Filename stem mismatch: {Path(lr).name} vs {Path(gt).name}"
        self.file_pairs = list(zip(self.lr_filenames, self.gt_filenames))

    def __len__(self):
        return len(self.file_pairs)

    def _augment(self, lr, gt):
        if random.random() > 0.5:
            lr, gt = np.fliplr(lr), np.fliplr(gt)
        if random.random() > 0.5:
            lr, gt = np.flipud(lr), np.flipud(gt)
        k = random.randint(0, 3)
        if k > 0:
            lr, gt = np.rot90(lr, k=k), np.rot90(gt, k=k)
        return lr.copy(), gt.copy()

    def __getitem__(self, idx):
        lr_path, gt_path = self.file_pairs[idx]
        lr_arr = np.load(lr_path).astype(np.float32)
        gt_arr = np.clip(np.load(gt_path).astype(np.float32), 0, 1)

        if self.augment:
            lr_arr, gt_arr = self._augment(lr_arr, gt_arr)

        lr_tensor = torch.from_numpy(lr_arr).unsqueeze(0)
        gt_tensor = torch.from_numpy(gt_arr).unsqueeze(0)
        return lr_tensor, gt_tensor

# Create Train & Validation Split
full_dataset = PairedNpyDataset(LR_DIR, GT_DIR, augment=True)
num_total = len(full_dataset)
num_val = max(1, int(num_total * 0.15))
num_train = num_total - num_val

train_ds, val_ds = random_split(full_dataset, [num_train, num_val], generator=torch.Generator().manual_seed(SEED))
val_ds.dataset.augment = False

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

print(f"[INFO] Dataset Split: {num_train} Training samples, {num_val} Validation samples.")

[INFO] Dataset Split: 2720 Training samples, 480 Validation samples.


## 5. Model Configuration & Architecture
NAFNet Baseline for 	imes$ Joint Super-Resolution and Denoising.

In [12]:
class LayerNorm2d(nn.Module):
    def __init__(self, channels: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(channels))
        self.bias = nn.Parameter(torch.zeros(channels))
        self.eps = eps

    def forward(self, x):
        u = x.mean(1, keepdim=True)
        s = (x - u).pow(2).mean(1, keepdim=True)
        x = (x - u) / torch.sqrt(s + self.eps)
        return self.weight.unsqueeze(-1).unsqueeze(-1) * x + self.bias.unsqueeze(-1).unsqueeze(-1)

class SimpleGate(nn.Module):
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2

class NAFBlock(nn.Module):
    def __init__(self, c: int, dw_expand: int = 2, ffn_expand: int = 2):
        super().__init__()
        dw_channel = c * dw_expand
        self.conv1 = nn.Conv2d(c, dw_channel, 1)
        self.conv2 = nn.Conv2d(dw_channel, dw_channel, 3, padding=1, groups=dw_channel)
        self.sg1 = SimpleGate()
        self.sca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(dw_channel // 2, dw_channel // 2, 1)
        )
        self.conv3 = nn.Conv2d(dw_channel // 2, c, 1)

        ffn_channel = c * ffn_expand
        self.conv4 = nn.Conv2d(c, ffn_channel, 1)
        self.sg2 = SimpleGate()
        self.conv5 = nn.Conv2d(ffn_channel // 2, c, 1)

        self.norm1 = LayerNorm2d(c)
        self.norm2 = LayerNorm2d(c)
        self.beta = nn.Parameter(torch.zeros((1, c, 1, 1)), requires_grad=True)
        self.gamma = nn.Parameter(torch.zeros((1, c, 1, 1)), requires_grad=True)

    def forward(self, x):
        res = x
        x = self.norm1(x)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.sg1(x)
        x = x * self.sca(x)
        x = self.conv3(x)
        y = res + x * self.beta

        res = y
        y = self.norm2(y)
        y = self.conv4(y)
        y = self.sg2(y)
        y = self.conv5(y)
        return res + y * self.gamma

class NAFNetSR(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, width=32, enc_blocks=[2, 2], middle_blocks=4, dec_blocks=[2, 2], upscale=2):
        super().__init__()
        self.upscale = upscale
        self.intro = nn.Conv2d(in_channels, width, 3, padding=1)

        self.encoders = nn.ModuleList()
        self.downs = nn.ModuleList()
        curr_width = width
        for n in enc_blocks:
            self.encoders.append(nn.Sequential(*[NAFBlock(curr_width) for _ in range(n)]))
            self.downs.append(nn.Conv2d(curr_width, curr_width * 2, 2, stride=2))
            curr_width *= 2

        self.middle = nn.Sequential(*[NAFBlock(curr_width) for _ in range(middle_blocks)])

        self.decoders = nn.ModuleList()
        self.ups = nn.ModuleList()
        for n in dec_blocks:
            self.ups.append(nn.Sequential(
                nn.Conv2d(curr_width, curr_width * 2, 1),
                nn.PixelShuffle(2)
            ))
            curr_width = curr_width // 2
            self.decoders.append(nn.Sequential(*[NAFBlock(curr_width) for _ in range(n)]))

        self.sr_upsample = nn.Sequential(
            nn.Conv2d(curr_width, curr_width * (upscale ** 2), 3, padding=1),
            nn.PixelShuffle(upscale),
            NAFBlock(curr_width)
        )
        self.ending = nn.Conv2d(curr_width, out_channels, 3, padding=1)

    def forward(self, x):
        feats = self.intro(x)
        skips = []
        for encoder, down in zip(self.encoders, self.downs):
            feats = encoder(feats)
            skips.append(feats)
            feats = down(feats)

        feats = self.middle(feats)

        for up, decoder in zip(self.ups, self.decoders):
            feats = up(feats)
            feats = feats + skips.pop()
            feats = decoder(feats)

        feats = self.sr_upsample(feats)
        return self.ending(feats)

model = NAFNetSR(in_channels=1, out_channels=1, width=32).to(device)
print(f"[INFO] Model '{MODEL_NAME}' Initialized. Total Trainable Parameters: {sum(p.numel() for p in model.parameters()):,}")

[INFO] Model 'nafnet' Initialized. Total Trainable Parameters: 760,001


## 6. Loss Function, Evaluation Metrics & Optimizer

In [ ]:
class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps = eps
    def forward(self, pred, target):
        return torch.mean(torch.sqrt((pred - target)**2 + self.eps**2))

def calculate_psnr(pred: torch.Tensor, target: torch.Tensor, max_val: float = 1.0) -> float:
    pred = torch.clamp(pred, 0.0, max_val)
    target = torch.clamp(target, 0.0, max_val)
    mse = torch.mean((pred - target) ** 2).item()
    if mse == 0:
        return float('inf')
    return 20.0 * math.log10(max_val) - 10.0 * math.log10(mse)

def calculate_ssim(pred: torch.Tensor, target: torch.Tensor, max_val: float = 1.0, window_size: int = 11) -> float:
    if pred.dim() == 2:
        pred = pred.unsqueeze(0).unsqueeze(0)
        target = target.unsqueeze(0).unsqueeze(0)
    elif pred.dim() == 3:
        pred = pred.unsqueeze(0)
        target = target.unsqueeze(0)
        
    # Cast to float32: AMP autocast leaves tensors as float16 (HalfTensor).
    # float16 max ~65504 — squaring (mu1**2, pred*pred) overflows to inf.
    # float32 max ~3.4e38 handles SSIM arithmetic safely with no accuracy loss.
    pred = pred.float()
    target = target.float()
    
    pred = torch.clamp(pred, 0.0, max_val)
    target = torch.clamp(target, 0.0, max_val)
    
    C1 = (0.01 * max_val) ** 2
    C2 = (0.03 * max_val) ** 2
    
    sigma = 1.5
    gauss = torch.exp(-torch.arange(window_size).float().sub(window_size // 2).pow(2) / (2 * sigma ** 2))
    kernel_1d = gauss / gauss.sum()
    kernel_2d = kernel_1d.unsqueeze(1) @ kernel_1d.unsqueeze(0)
    # Match kernel to input device AND dtype (critical for AMP half-precision inputs)
    kernel = kernel_2d.expand(pred.size(1), 1, window_size, window_size).to(device=pred.device, dtype=pred.dtype)
    
    pad = window_size // 2
    mu1 = F.conv2d(pred, kernel, padding=pad, groups=pred.size(1))
    mu2 = F.conv2d(target, kernel, padding=pad, groups=target.size(1))
    
    mu1_sq = mu1.pow(2)
    mu2_sq = mu2.pow(2)
    mu1_mu2 = mu1 * mu2
    
    sigma1_sq = F.conv2d(pred * pred, kernel, padding=pad, groups=pred.size(1)) - mu1_sq
    sigma2_sq = F.conv2d(target * target, kernel, padding=pad, groups=target.size(1)) - mu2_sq
    sigma12 = F.conv2d(pred * target, kernel, padding=pad, groups=pred.size(1)) - mu1_mu2
    
    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    return ssim_map.mean().item()

criterion = CharbonnierLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))


## 7. Training & Validation Loop (Auto-Saves to )

In [ ]:
train_losses, val_losses, val_psnrs, val_ssims = [], [], [], []
best_psnr = 0.0

print(f"[INFO] Starting Training Loop for '{MODEL_NAME}' (Version: '{MODEL_VERSION}')...")
epoch_bar = tqdm(range(1, EPOCHS + 1), desc="Overall Progress")

for epoch in epoch_bar:
    # Train Epoch
    model.train()
    running_loss = 0.0
    train_loader_tqdm = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS:02d} [Train]", leave=False)
    for lr_imgs, gt_imgs in train_loader_tqdm:
        lr_imgs, gt_imgs = lr_imgs.to(device), gt_imgs.to(device)
        optimizer.zero_grad(set_to_none=True)
        
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            pred_imgs = model(lr_imgs)
            loss = criterion(pred_imgs, gt_imgs)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        
        batch_loss = loss.item()
        running_loss += batch_loss * lr_imgs.size(0)
        train_loader_tqdm.set_postfix({'batch_loss': f'{batch_loss:.4f}'})
        
    scheduler.step()
    train_loss = running_loss / len(train_loader.dataset)
    train_losses.append(train_loss)
    
    # Validation Epoch
    model.eval()
    val_loss, val_psnr, val_ssim = 0.0, 0.0, 0.0
    val_loader_tqdm = tqdm(val_loader, desc=f"Epoch {epoch:02d}/{EPOCHS:02d} [Val]", leave=False)
    with torch.no_grad():
        for lr_imgs, gt_imgs in val_loader_tqdm:
            lr_imgs, gt_imgs = lr_imgs.to(device), gt_imgs.to(device)
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                pred_imgs = model(lr_imgs)
                loss = criterion(pred_imgs, gt_imgs)
            val_loss += loss.item() * lr_imgs.size(0)
            for p, g in zip(pred_imgs, gt_imgs):
                val_psnr += calculate_psnr(p, g)
                val_ssim += calculate_ssim(p, g)
                
    val_loss /= len(val_loader.dataset)
    val_psnr /= len(val_loader.dataset)
    val_ssim /= len(val_loader.dataset)
    val_losses.append(val_loss)
    val_psnrs.append(val_psnr)
    val_ssims.append(val_ssim)
    
    if val_psnr > best_psnr:
        best_psnr = val_psnr
        torch.save(model.state_dict(), CHECKPOINT_BEST_PATH)
        torch.save(model.state_dict(), VERSION_BEST_SAVE_PATH)
        
    tqdm.write(f"Epoch {epoch:02d}/{EPOCHS:02d} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | Val PSNR: {val_psnr:.2f} dB | Val SSIM: {val_ssim:.4f} (Best PSNR: {best_psnr:.2f} dB)")

# Save Final Model Weights (Latest & Versioned)
torch.save(model.state_dict(), CHECKPOINT_FINAL_PATH)
torch.save(model.state_dict(), VERSION_FINAL_SAVE_PATH)
torch.save(model.state_dict(), VERSION_SAVE)
print(f"[SUCCESS] Saved latest best weights to '{CHECKPOINT_BEST_PATH}'")
print(f"[SUCCESS] Saved latest final weights to '{CHECKPOINT_FINAL_PATH}'")
print(f"[SUCCESS] Saved versioned best weights to '{VERSION_BEST_SAVE_PATH}'")
print(f"[SUCCESS] Saved versioned final weights to '{VERSION_FINAL_SAVE_PATH}'")

[INFO] Starting Training Loop for 'nafnet' (Version: 'nafnet1')...


Overall Progress:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 01/15 [Train]:   0%|          | 0/340 [00:00<?, ?it/s]

Epoch 01/15 [Val]:   0%|          | 0/60 [00:00<?, ?it/s]

Epoch 01/15 | Train Loss: 0.046535 | Val Loss: 0.037838 | Val PSNR: 26.55 dB | Val SSIM: 0.6716 (Best PSNR: 26.55 dB)


...

Epoch 10/15 | Train Loss: 0.030916 | Val Loss: 0.032822 | Val PSNR: 27.93 dB | Val SSIM: 0.7561 (Best PSNR: 28.29 dB)


Epoch 11/15 [Train]:   0%|          | 0/340 [00:00<?, ?it/s]

Epoch 11/15 [Val]:   0%|          | 0/60 [00:00<?, ?it/s]

Epoch 11/15 | Train Loss: 0.030660 | Val Loss: 0.030997 | Val PSNR: 28.42 dB | Val SSIM: 0.7590 (Best PSNR: 28.42 dB)


Epoch 12/15 [Train]:   0%|          | 0/340 [00:00<?, ?it/s]

## 8. Visualization & Performance Curves

In [15]:
# Plot Loss, PSNR & SSIM Curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(range(1, EPOCHS + 1), train_losses, label="Train Loss", color='blue')
axes[0].plot(range(1, EPOCHS + 1), val_losses, label="Val Loss", color='orange')
axes[0].set_title(f"{MODEL_NAME} — Charbonnier Loss Curve")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True)
axes[0].legend()

axes[1].plot(range(1, EPOCHS + 1), val_psnrs, label="Val PSNR", color='green')
axes[1].set_title(f"{MODEL_NAME} — Validation PSNR (dB)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("PSNR (dB)")
axes[1].grid(True)
axes[1].legend()

axes[2].plot(range(1, EPOCHS + 1), val_ssims, label="Val SSIM", color='purple')
axes[2].set_title(f"{MODEL_NAME} — Validation SSIM")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("SSIM")
axes[2].grid(True)
axes[2].legend()

plt.tight_layout()
plt.show()

## 9. Qualitative Results Comparison
Side-by-side comparison of **Input (NoisyLR 128x128)**, **Restored Output (256x256)**, **Ground Truth (GT 256x256)**, **Absolute Error Map |Prediction - GT|**, and **Residual Error Map (Prediction - GT)**.

In [ ]:
# Load best model weights
if os.path.exists(CHECKPOINT_BEST_PATH):
    model.load_state_dict(torch.load(CHECKPOINT_BEST_PATH, map_location=device))
    print(f"[INFO] Loaded best model checkpoint from '{CHECKPOINT_BEST_PATH}'.")

model.eval()
with torch.no_grad():
    lr_batch, gt_batch = next(iter(val_loader))
    lr_batch, gt_batch = lr_batch.to(device), gt_batch.to(device)
    pred_batch = model(lr_batch)

num_display = min(3, lr_batch.size(0))
fig, axes = plt.subplots(num_display, 5, figsize=(22, 4.5 * num_display))

for i in range(num_display):
    lr_img = lr_batch[i, 0].cpu().numpy()
    pred_tensor = torch.clamp(pred_batch[i, 0], 0.0, 1.0)
    gt_tensor = gt_batch[i, 0]
    
    pred_img = pred_tensor.cpu().numpy()
    gt_img = gt_tensor.cpu().numpy()
    
    abs_err = np.abs(pred_img - gt_img)
    residual_err = pred_img - gt_img
    
    psnr_score = calculate_psnr(pred_batch[i], gt_batch[i])
    ssim_score = calculate_ssim(pred_batch[i], gt_batch[i])
    
    ax_row = axes[i] if num_display > 1 else axes
    
    # 1. Input NoisyLR
    im0 = ax_row[0].imshow(lr_img, cmap='gray')
    ax_row[0].set_title(f"Sample {i+1}: Input NoisyLR (128x128)")
    ax_row[0].axis('off')
    
    # 2. Restored NAFNet
    im1 = ax_row[1].imshow(pred_img, cmap='gray', vmin=0, vmax=1)
    ax_row[1].set_title(f"Restored {MODEL_NAME.upper()} (256x256) PSNR: {psnr_score:.2f} dB | SSIM: {ssim_score:.4f}")
    ax_row[1].axis('off')
    
    # 3. Ground Truth GT
    im2 = ax_row[2].imshow(gt_img, cmap='gray', vmin=0, vmax=1)
    ax_row[2].set_title(f"Ground Truth GT (256x256)")
    ax_row[2].axis('off')
    
    # 4. Absolute Error Map |pred - gt|
    im3 = ax_row[3].imshow(abs_err, cmap='inferno')
    ax_row[3].set_title(f"Absolute Error Map|Pred - GT|")
    ax_row[3].axis('off')
    plt.colorbar(im3, ax=ax_row[3], fraction=0.046, pad=0.04)
    
    # 5. Residual Error Map (pred - gt)
    im4 = ax_row[4].imshow(residual_err, cmap='coolwarm', vmin=-0.5, vmax=0.5)
    ax_row[4].set_title(f"Residual Error Map(Pred - GT)")
    ax_row[4].axis('off')
    plt.colorbar(im4, ax=ax_row[4], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## 10. Interactive Single-Sample Visualizer & Signal Inspector

Execute the cell below to interactively inspect any sample by filename (e.g. `0001`, `image_0042.npy`, or relative stem).
It will perform model inference, calculate PSNR/SSIM, and generate a comprehensive 2-row analysis dashboard (Image Comparison, Error Heatmaps, Center Intensity Profile, and Pixel Histograms).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 10. INTERACTIVE SINGLE-SAMPLE VISUALIZER & INSPECTOR
# ─────────────────────────────────────────────────────────────────────────────

def inspect_sample_by_name():
    # Ensure model is in evaluation mode
    model.eval()
    
    # List available files for helpful prompting
    all_lr_files = sorted(glob.glob(os.path.join(LR_DIR, "*.npy")))
    if not all_lr_files:
        print(f"[ERROR] No .npy files found in LR directory: {LR_DIR}")
        return
    
    sample_stems = [Path(f).stem for f in all_lr_files]
    print(f"[INFO] Found {len(all_lr_files)} samples in dataset.")
    print(f"[INFO] Example valid inputs: '{sample_stems[0]}', '{sample_stems[1]}', '{sample_stems[min(10, len(sample_stems)-1)]}'")
    
    # Prompt user for filename
    user_input = input("\nEnter sample filename or stem (e.g. '0001' or '0001.npy') [Press Enter for default]: ").strip()
    
    if not user_input:
        target_stem = sample_stems[0]
        print(f"[INFO] No input provided. Defaulting to first sample: '{target_stem}'")
    else:
        # Clean user input (remove extension if present)
        clean_input = Path(user_input).stem
        if clean_input in sample_stems:
            target_stem = clean_input
        else:
            # Try partial matching
            matches = [s for s in sample_stems if clean_input in s]
            if matches:
                target_stem = matches[0]
                print(f"[INFO] Matched '{user_input}' to dataset sample '{target_stem}'")
            else:
                print(f"[ERROR] Could not find sample matching '{user_input}'. Available range: '{sample_stems[0]}' to '{sample_stems[-1]}'.")
                return
    
    # Construct exact file paths
    lr_path = os.path.join(LR_DIR, f"{target_stem}.npy")
    gt_path = os.path.join(GT_DIR, f"{target_stem}.npy")
    
    if not os.path.exists(lr_path) or not os.path.exists(gt_path):
        print(f"[ERROR] Pair files do not exist:\n  LR: {lr_path}\n  GT: {gt_path}")
        return
    
    # Load numpy arrays
    lr_arr = np.load(lr_path).astype(np.float32)
    gt_arr = np.clip(np.load(gt_path).astype(np.float32), 0.0, 1.0)
    
    # Convert to PyTorch Tensor & Run Inference
    lr_tensor = torch.from_numpy(lr_arr).unsqueeze(0).unsqueeze(0).to(device)
    gt_tensor = torch.from_numpy(gt_arr).unsqueeze(0).unsqueeze(0).to(device)
    
    with torch.no_grad():
        pred_tensor = model(lr_tensor)
        pred_tensor = torch.clamp(pred_tensor, 0.0, 1.0)
        
    # Calculate Metrics
    psnr_val = calculate_psnr(pred_tensor[0], gt_tensor[0])
    ssim_val = calculate_ssim(pred_tensor[0], gt_tensor[0])
    
    # Extract 2D numpy arrays for plotting
    pred_arr = pred_tensor[0, 0].cpu().numpy()
    
    abs_err = np.abs(pred_arr - gt_arr)
    residual_err = pred_arr - gt_arr
    
    # ─────────────────────────────────────────────────────────────────────────
    # DASHBOARD PLOTTING (2 Rows)
    # ─────────────────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(22, 10))
    gs = fig.add_gridspec(2, 5, height_ratios=[1.2, 1.0])
    
    # Row 1: Main Spatial Visualizations
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[0, 2])
    ax4 = fig.add_subplot(gs[0, 3])
    ax5 = fig.add_subplot(gs[0, 4])
    
    # 1. Input NoisyLR
    im1 = ax1.imshow(lr_arr, cmap='gray')
    ax1.set_title(f"Input NoisyLR\n({target_stem}) — {lr_arr.shape}", fontsize=11, fontweight='bold')
    ax1.axis('off')
    plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
    
    # 2. Restored NAFNet Output
    im2 = ax2.imshow(pred_arr, cmap='gray', vmin=0.0, vmax=1.0)
    ax2.set_title(f"Restored {MODEL_NAME.upper()}\nPSNR: {psnr_val:.2f} dB | SSIM: {ssim_val:.4f}", fontsize=11, fontweight='bold', color='darkgreen')
    ax2.axis('off')
    plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
    
    # 3. Ground Truth GT
    im3 = ax3.imshow(gt_arr, cmap='gray', vmin=0.0, vmax=1.0)
    ax3.set_title(f"Ground Truth GT\n{gt_arr.shape}", fontsize=11, fontweight='bold')
    ax3.axis('off')
    plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)
    
    # 4. Absolute Error Map
    im4 = ax4.imshow(abs_err, cmap='inferno')
    ax4.set_title(f"Absolute Error Map\n|Pred - GT|", fontsize=11, fontweight='bold')
    ax4.axis('off')
    plt.colorbar(im4, ax=ax4, fraction=0.046, pad=0.04)
    
    # 5. Residual Error Map
    vmax_res = max(0.1, np.percentile(np.abs(residual_err), 99))
    im5 = ax5.imshow(residual_err, cmap='coolwarm', vmin=-vmax_res, vmax=vmax_res)
    ax5.set_title(f"Residual Map (Pred - GT)\nRange: [{-vmax_res:.2f}, {vmax_res:.2f}]", fontsize=11, fontweight='bold')
    ax5.axis('off')
    plt.colorbar(im5, ax=ax5, fraction=0.046, pad=0.04)
    
    # Row 2: Signal & Statistical Analysis
    ax_line = fig.add_subplot(gs[1, 0:3])
    ax_hist = fig.add_subplot(gs[1, 3:5])
    
    # Center Row Intensity Cross-Section Profile
    mid_row = gt_arr.shape[0] // 2
    ax_line.plot(pred_arr[mid_row, :], label="Restored NAFNet", color="crimson", linewidth=2.0)
    ax_line.plot(gt_arr[mid_row, :], label="Ground Truth GT", color="black", linestyle="--", linewidth=1.8, alpha=0.8)
    
    # Upsample LR center row for direct profile comparison
    lr_mid_row = lr_arr[lr_arr.shape[0] // 2, :]
    lr_x = np.linspace(0, gt_arr.shape[1] - 1, num=len(lr_mid_row))
    ax_line.plot(lr_x, lr_mid_row, label="Input NoisyLR (Interpolated)", color="steelblue", linestyle=":", alpha=0.7)
    
    ax_line.set_title(f"Center Row Intensity Cross-Section Profile (Row Y={mid_row})", fontsize=12, fontweight='bold')
    ax_line.set_xlabel("Pixel Column (X)")
    ax_line.set_ylabel("Normalized Intensity")
    ax_line.grid(True, alpha=0.3)
    ax_line.legend(loc="upper right", frameon=True)
    
    # Pixel Intensity Histogram Comparison
    ax_hist.hist(gt_arr.ravel(), bins=50, alpha=0.5, color="black", label="GT Pixel Dist", density=True)
    ax_hist.hist(pred_arr.ravel(), bins=50, alpha=0.5, color="crimson", label="Restored Pixel Dist", density=True)
    ax_hist.set_title("Pixel Value Distribution Comparison", fontsize=12, fontweight='bold')
    ax_hist.set_xlabel("Pixel Value")
    ax_hist.set_ylabel("Density")
    ax_hist.grid(True, alpha=0.3)
    ax_hist.legend(loc="upper right", frameon=True)
    
    plt.suptitle(f"Sample Inspection Dashboard — Stem: '{target_stem}'", fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()

# Run Interactive Function
inspect_sample_by_name()
